<img src="https://www.plus2net.com/images/top2.jpg" alt="plus2net logo" width="400">

# Python MySQL Automated Backup and Restore Practice in Google Colab

**Main Tutorial Link:** [Python MySQL Backup Tutorial](https://www.plus2net.com/python/mysql-backup.php)

---

## 1. Environment Setup & Local MySQL/MariaDB Server Daemon

Install MariaDB server and initialize the local daemon within the Google Colab environment.

In [ ]:
# Stop and clean up any previously running MariaDB/MySQL instances completely
!pkill -9 mariadbd || true
!pkill -9 mysqld || true
!pkill -9 mysqld_safe || true
!rm -rf /var/run/mysqld/* /var/lib/mysql/* /run/mysqld/* || true

# Install MariaDB server and pymysql drivers
!apt-get update && apt-get install -y mariadb-server > /dev/null
!pip install pymysql cryptography > /dev/null

# Ensure run directories exist with proper system permissions
!mkdir -p /run/mysqld /var/run/mysqld /var/lib/mysql
!chown -R mysql:mysql /run/mysqld /var/run/mysqld /var/lib/mysql
!chmod 777 /run/mysqld /var/run/mysqld

# Initialize default database schema
!mysql_install_db --user=mysql --basedir=/usr --datadir=/var/lib/mysql > /dev/null

# Start MariaDB daemon in the background
import subprocess
import time
import os

print("Starting MariaDB daemon process...")
subprocess.Popen([
    "/usr/sbin/mariadbd",
    "--user=mysql",
    "--pid-file=/run/mysqld/mysqld.pid",
    "--socket=/run/mysqld/mysqld.sock",
    "--datadir=/var/lib/mysql",
    "--skip-networking=0"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Dynamically wait and poll for the socket file to become active
success = False
for i in range(15):
    if os.path.exists("/run/mysqld/mysqld.sock"):
        success = True
        print(f"Database socket found active after {i} seconds!")
        break
    time.sleep(1)

if not success:
    print("Warning: Database socket not found yet. Attempting setup anyway...")

# Create database and configure dedicated credentials via administrative socket connection
!mysql --socket=/run/mysqld/mysqld.sock -u root -e "CREATE DATABASE IF NOT EXISTS my_tutorial; \
  CREATE USER IF NOT EXISTS 'demo_user'@'localhost' IDENTIFIED BY 'demo_password'; \
  GRANT ALL PRIVILEGES ON my_tutorial.* TO 'demo_user'@'localhost'; \
  FLUSH PRIVILEGES;"

print("MySQL/MariaDB server running and user 'demo_user' configured successfully!")

## 2. Download SQL Dump (`student.sql`) & Import Database

Fetch `student.sql` directly from plus2net and import it into the local MySQL database.

In [ ]:
# Download the student.sql dump file into the Colab environment
!wget -q -O student.sql https://www.plus2net.com/python/download/student.sql || echo "Note: Ensure student.sql is hosted at this path"

# Import student.sql into the my_tutorial database
!mysql --socket=/run/mysqld/mysqld.sock -u root my_tutorial < student.sql

print("Database table 'student' created and populated successfully via student.sql!")

## 3. Automated Backup Using `mariadb-dump` / `mysqldump`

Execute a database backup via Python subprocess using the active socket connection.

In [ ]:
import subprocess
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_filename = f"backup_my_tutorial_{timestamp}.sql"

dump_cmd = [
    "mariadb-dump",
    "--socket=/run/mysqld/mysqld.sock",
    "-u", "root",
    "my_tutorial"
]

try:
    with open(backup_filename, "w") as f:
        subprocess.run(dump_cmd, stdout=f, check=True)
    print(f"Database backup created successfully: {backup_filename}")
    print(f"Backup file size: {os.path.getsize(backup_filename)} bytes")
except subprocess.CalledProcessError as e:
    print("Backup failed:", e)

## 4. Restoring Database from Backup File

Simulate dropping the table and restoring it from the generated backup file.

In [ ]:
import pymysql

# Drop table to simulate data loss
conn = pymysql.connect(user='root', database='my_tutorial', unix_socket='/run/mysqld/mysqld.sock')
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS student")
conn.commit()
conn.close()
print("Table 'student' dropped for restoration test.")

# Restore from backup file via subprocess
restore_cmd = [
    "mysql",
    "--socket=/run/mysqld/mysqld.sock",
    "-u", "root",
    "my_tutorial"
]

with open(backup_filename, "r") as f:
    subprocess.run(restore_cmd, stdin=f, check=True)

# Verify restoration
connn = pymysql.connect(user='root', database='my_tutorial', unix_socket='/run/mysqld/mysqld.sock')
cursorr = connn.cursor()
cursorr.execute("SELECT COUNT(*) FROM student")
count = cursorr.fetchone()[0]
connn.close()
print(f"Database restored successfully! Total rows in student table: {count}")